# 3 What does the algorithm know?

In lesson 1 (`04_search`) you handed a search space to Optuna's TPE and read what came back.
TPE was one tool there, used without looking inside. This notebook looks inside. Optuna
calls the part of a search that picks the next trial the **sampler**: it gets the space and
the trials so far, and returns the values the next trial will train with. Samplers differ in
one thing only, what they *know* when they pick.

At the end of this notebook you can say, for four samplers, what each one knows and which
shortcoming of the one before it removes:

| sampler | what it knows when it picks a trial |
|---|---|
| `GridSampler` | a list of values per axis, nothing about the trials so far |
| `RandomSampler` | the distribution of every axis, nothing about the trials so far |
| `TPESampler` | which of the trials so far scored well and which did not |
| `NSGAIISampler` | a population: the best few trials so far, which it recombines |

You also learn to read a search by the shape of its best-so-far curve, to ask first whether
the space has anything in it to find, and to report the number of a winner from data the
search never saw. That last one is the number your class problem asks for.

The data is DTD (describable textures) from lesson 1, on cached vectors of three encoders,
so a trial costs a fraction of a second and a search of 40 trials a few seconds. That is
what makes it affordable to run every sampler several times and look at the spread.

In [ ]:
import sys
import time

import datasets
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from loguru import logger
from notebooktester import param
from torch import nn
from vectormesh import VectorCache
from vectormesh.components import NeuralNet

from hpo.measure import Scalars
from hpo.record import RunRecord, Store, StudyLog
from hpo.search import Choice, Float, Space

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
optuna.logging.set_verbosity(optuna.logging.WARNING)

BUDGET = param(40, test=3)  # trials per search
SEARCH_EPOCHS = param(5, test=1)  # epochs per trial
SEEDS = param(tuple(range(6)), test=(0, 1))  # one search per sampler per seed
FINAL_EPOCHS = param(30, test=2)  # the longest retrain of the winner in 3.6
BATCH, N_CLASSES = 128, 47
ENCODERS = ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"]

SPLITS = {}
for enc in ENCODERS:
    cache = VectorCache.from_hub(f"pttrn-io/dtd-{enc}")
    # the cache is stored grouped by label: shuffle before slicing, the same order for
    # every encoder, so row i is the same image in all three
    d = cache.dataset.shuffle(seed=0).with_format("numpy")
    X = torch.tensor(np.asarray(d["embed"], dtype="float32"))
    y = torch.tensor(np.asarray(d["label"]))
    a, b = int(0.60 * len(X)), int(0.80 * len(X))
    SPLITS[enc] = {
        "train": (X[:a], y[:a]),
        "val": (X[a:b], y[a:b]),
        "test": (X[b:], y[b:]),
    }

print({split: len(xy[0]) for split, xy in SPLITS["dinov2-small"].items()})
print(
    {enc: SPLITS[enc]["train"][0].shape[1] for enc in ENCODERS},
    "dimensions per encoder",
)

**A three-way split, and here it matters more than anywhere else.** A search tries dozens
of configurations and keeps the best one. The winner's score is therefore the *maximum* of
dozens of noisy numbers, and a maximum of noisy numbers lies above what the same
configuration scores on a fresh run. The search chooses on **validation**; the winner is
scored on **test**, once, at the end (§3.6). The three splits come from the same shuffled
3,760 rows of the DTD cache, in the same order for every encoder.

### One trial: a head on cached vectors

Every trial trains the head of lesson 1 on the vectors of one encoder: `NeuralNet` with
`depth` hidden layers of `width` units (GELU between them) and 47 class scores out, Adam
with learning rate `lr`, batches of 128. `train_head` returns the validation accuracy after
every epoch, so a trial can report its whole curve to the study, as lesson 1's search did.

Every trial trains **5 epochs**. That is short on purpose: a search is a *ranking*, and
every candidate gets the same short budget, so the comparison between them stays equal. For
a number you **report** the length has to be justified, and §3.6 measures how much the
length changes the winner's score.

In [ ]:
def train_head(
    config: dict, seed: int, epochs: int, on=("val",)
) -> dict[str, list[float]]:
    """Train the head `config` describes for `epochs` epochs; return its accuracy after
    every epoch on each split in `on`."""
    Xtr, ytr = SPLITS[config["encoder"]]["train"]
    torch.manual_seed(seed)
    model = NeuralNet([Xtr.shape[1], *[config["width"]] * config["depth"]], N_CLASSES)
    opt = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.CrossEntropyLoss()
    accuracy = {split: [] for split in on}
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr) - 1, BATCH):
            idx = perm[i : i + BATCH]
            opt.zero_grad()
            loss_fn(model(Xtr[idx]), ytr[idx]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            for split in on:
                Xe, ye = SPLITS[config["encoder"]][split]
                accuracy[split].append(
                    float((model(Xe).argmax(1) == ye).float().mean())
                )
    return accuracy

## 3.1 Grid: list the values, try the combinations

The sampler that knows least. You give it a list of values per axis, the **lattice**, and
it tries combinations of them. It never looks at a score: trial 30 is chosen exactly as
trial 1 was. Optuna's `GridSampler` walks the combinations in a shuffled order (the seed
sets the order), so a budget smaller than the grid gets a random subset of the lattice.

Below are the two spaces of this notebook, written as `hpo.search.Space` as in lesson 1, and
the grid's lattice for each. Round 1 keeps the encoder fixed to `dinov2-small` and searches
the head; round 2 puts the encoder in the space as well.

**Before you run it:** how many lattice points does each space have, and what part of them
does a budget of 40 trials reach?

In [ ]:
WIDTHS, DEPTHS = [64, 128, 256, 512, 1024], [1, 2, 3]

SPACE = {
    "head": Space(
        {
            "encoder": "dinov2-small",  # fixed: every trial gets it
            "width": Choice(WIDTHS),
            "depth": Choice(DEPTHS),
            "lr": Float(1e-4, 1e-2, log=True),
            "epochs": SEARCH_EPOCHS,
        }
    ),
    "encoder": Space(
        {
            "encoder": Choice(ENCODERS),
            "width": Choice(WIDTHS),
            "depth": Choice(DEPTHS),
            "lr": Float(1e-4, 1e-2, log=True),
            "epochs": SEARCH_EPOCHS,
        }
    ),
}

# the grid's lattice: the values it may draw on every axis the space searches
LATTICE = {
    "encoder": ENCODERS,
    "width": WIDTHS,
    "depth": DEPTHS,
    "lr": [1e-4, 1e-3, 1e-2],
}

for name, space in SPACE.items():
    lattice = {axis: LATTICE[axis] for axis in space.searched()}
    n = int(np.prod([len(values) for values in lattice.values()]))
    print(f"space '{name}': {space.line()}")
    print(f"  grid lattice: {lattice}")
    print(
        f"  {n} points; a budget of {BUDGET} trials visits {min(BUDGET / n, 1):.0%} of them\n"
    )

Grid's cost is the **product** of the axes: one extra axis with three values multiplies the
lattice by three, while the budget stays what it is. That is the shortcoming the next
sampler removes.

Note also what grid *can* draw: three learning rates, `1e-4`, `1e-3` and `1e-2`. If the best
learning rate is `3e-3`, grid cannot find it, however long it runs. The other three samplers
draw `lr` anywhere on the log scale between `1e-4` and `1e-2`.

## 3.2 Random: draw instead of list

`RandomSampler` knows the distribution of every axis and nothing else: every trial draws
every axis afresh, independently of all trials before it. Bergstra and Bengio (2012) gave
the argument that made it the default over grid: when only some axes matter, a grid
re-tests the same few values of the axis that matters at every value of the axes that do
not, while random gives every trial a new value of *every* axis. Forty trials of random give
forty different learning rates; forty trials of grid give three.

What random does not know is what it has seen. After thirty trials in which every `lr`
above `5e-3` scored badly, it still draws there as often as anywhere else.

## 3.3 TPE: a model of where the good trials were

`TPESampler` (Tree-structured Parzen Estimator) is the one you used in lesson 1. It removes
random's blindness. After every trial it splits the trials so far into the best few (by
default about the top tenth) and the rest, and estimates per axis how the values of the
good trials are spread and how those of the rest are spread. It then draws a number of
candidates and runs the one whose values are most typical of the good trials and least
typical of the rest. Its first 10 trials are random (`n_startup_trials=10`): it needs trials
before it has anything to model. With the same seed they are the very trials
`RandomSampler` draws, so TPE and random part ways only from trial 11 on.

Where it fails: it can only model what the trials so far show. With 40 trials and 10 of them
random, it has 30 trials to learn from; on an axis that barely moves the score, the good
and the bad trials look the same and the model has nothing to go on.

## 3.4 A population: keep the best and recombine them

`NSGAIISampler` (NSGA-II, a genetic algorithm) knows something else again. It keeps a
**population**: the best `population_size` trials so far, here 8. It works in
**generations** of 8 trials. The first generation is random. Every next trial is a
**child** of two **parents** from the population: per axis it takes the value of one parent
or the other (this is **crossover**), and now and then it draws one axis afresh
(**mutation**), so values that no parent has can still come in. When a generation is done,
the population becomes the best 8 of the old population and the new children.

It was built to search for several objectives at once (that is the "non-dominated sorting"
in the name; notebook 04 reads such a front over score and parameters). With one objective,
"better" is simply "higher validation accuracy", and what is left is the population.

What it fixes in random: a good value found once is kept and passed on, instead of being
forgotten after its trial. Its first generation *is* a random search: with the same seed,
Optuna's `NSGAIISampler` draws its first 8 trials exactly as `RandomSampler` does, so the two
part ways only from trial 9 on. What it costs: it needs generations. A budget of 40 is five
generations of 8, which is little for recombination to work with; TPE learns from every
single trial, the population only between generations.

The four samplers, as Optuna objects. Every search gets its own seed (the sampler's seed and
the torch seed of every trial in it), and every sampler gets the same seeds, the same
space and the same budget. `search` writes each search to its own study,
`lesson3-algorithm-<round>-<sampler>-s<seed>`, through `StudyLog.search`, and every trial
reports its validation accuracy per epoch with `trial.report` and is recorded with
`StudyLog.record`, as in lesson 1. `make optuna-dashboard` shows them all.

In [ ]:
SAMPLERS = {
    "grid": lambda space, seed: optuna.samplers.GridSampler(
        {axis: LATTICE[axis] for axis in space.searched()}, seed=seed
    ),
    "random": lambda space, seed: optuna.samplers.RandomSampler(seed=seed),
    "tpe": lambda space, seed: optuna.samplers.TPESampler(seed=seed),
    "population": lambda space, seed: optuna.samplers.NSGAIISampler(
        population_size=8, seed=seed
    ),
}


def search(round_name: str, sampler: str, seed: int) -> StudyLog:
    """One search: `BUDGET` trials of `sampler` over the space of `round_name`, every
    trial trained with torch seed `seed`, written to its own study."""
    space = SPACE[round_name]
    name = f"lesson3-algorithm-{round_name}-{sampler}-s{seed}"
    # every run of the notebook starts the study afresh: a sampler that found earlier
    # trials in its study would start from them, and the budgets would not be equal
    if name in optuna.get_all_study_names(Store.default().url):
        optuna.delete_study(study_name=name, storage=Store.default().url)
    log = StudyLog.search(
        name=name,
        space=space,
        sampler=SAMPLERS[sampler](space, seed),
        attrs={
            "origin": "03_what_does_the_algorithm_know",
            "sampler": sampler,
            "seed": seed,
        },
    )

    def objective(trial: optuna.Trial) -> float:
        config = space.draw(trial)
        start = time.perf_counter()
        val = train_head(config, seed, config["epochs"])["val"]
        for epoch, accuracy in enumerate(val):
            trial.report(accuracy, epoch)
        log.record(
            trial,
            RunRecord(
                role="trial",
                dataset="dtd",
                config=config,
                fixed=space.fixed(),
                train_seconds=time.perf_counter() - start,
                extra={"sampler": sampler, "seed": seed},
            ),
        )
        return val[-1]

    log.study.optimize(objective, n_trials=BUDGET)
    return log


def race(round_name: str) -> dict[str, list[StudyLog]]:
    """Every sampler, once per seed, on the space of `round_name`."""
    start = time.time()
    logs = {s: [search(round_name, s, seed) for seed in SEEDS] for s in SAMPLERS}
    print(
        f"{len(SAMPLERS)} samplers x {len(SEEDS)} seeds x {BUDGET} trials "
        f"in {time.time() - start:.0f}s"
    )
    return logs


def values(log: StudyLog) -> np.ndarray:
    """The validation accuracy of every trial, in the order the trials ran."""
    return np.array([t.value for t in log.trials(optuna.trial.TrialState.COMPLETE)])


def best_table(logs: dict[str, list[StudyLog]]) -> pd.DataFrame:
    """Per sampler: the best validation accuracy of a search, over the seeds."""
    rows = []
    for sampler, per_seed in logs.items():
        best = np.array([values(log).max() for log in per_seed])
        rows.append(
            {
                "sampler": sampler,
                "mean": best.mean(),
                "std": best.std(ddof=1),
                "worst seed": best.min(),
                "best seed": best.max(),
            }
        )
    return pd.DataFrame(rows).set_index("sampler")

## 3.5 The race

### Round 1: the head, with the encoder fixed

**Question:** on the head-only space, does it matter which sampler searches? Each sampler
runs 40 trials, once per seed; the number per search is the best validation accuracy it
found.

**Before you run it, write down:** which sampler do you expect to find the best head, and by
how much more than the others? Think of what notebook 02 measured about the head's capacity
axis.

In [ ]:
round1 = race("head")
display(
    best_table(round1)
    .style.format("{:.4f}")
    .set_caption(
        f"round 1, head only: best validation accuracy of a search, over {len(SEEDS)} seeds"
    )
)

In [ ]:
def drawn(logs: dict[str, list[StudyLog]]) -> pd.DataFrame:
    """Per sampler, summed over its seeds: how many distinct values of lr it tried, and
    how many trials went to each encoder."""
    rows = []
    for sampler, per_seed in logs.items():
        trials = [
            t for log in per_seed for t in log.trials(optuna.trial.TrialState.COMPLETE)
        ]
        encoders = pd.Series([t.user_attrs["config"]["encoder"] for t in trials])
        rows.append(
            {
                "sampler": sampler,
                "trials": len(trials),
                "distinct lr": len({round(t.params["lr"], 8) for t in trials}),
                **{enc: int((encoders == enc).sum()) for enc in ENCODERS},
            }
        )
    return pd.DataFrame(rows).set_index("sampler")


def winners(logs: dict[str, list[StudyLog]]) -> pd.DataFrame:
    """The winning trial of every search: its validation accuracy and its config."""
    rows = []
    for sampler, per_seed in logs.items():
        for seed, log in zip(SEEDS, per_seed):
            best = log.study.best_trial
            rows.append(
                {
                    "sampler": sampler,
                    "seed": seed,
                    "val": best.value,
                    **{
                        k: best.user_attrs["config"][k]
                        for k in ("encoder", "width", "depth", "lr")
                    },
                }
            )
    return pd.DataFrame(rows)


display(
    drawn(round1).style.set_caption(
        "round 1: what each sampler drew, all seeds together"
    )
)

**Is a difference between two samplers larger than chance?** The best score of a search
moves from seed to seed, so a difference between two samplers means something only when it
is large compared to that movement. The measure, from lesson 1's `compare`, is the
**standard error of the difference**: with $n$ searches per sampler and spreads $s_1$ and
$s_2$ over the seeds, it is $\sqrt{s_1^2/n + s_2^2/n}$, the amount by which the difference
between the two means moves by chance alone. A difference of two standard errors or more is
unlikely to be chance; one of about one standard error is what chance gives all the time.
The table puts every sampler against random, the sampler that knows nothing about the
trials so far.

In [ ]:
def standard_error(a: np.ndarray, b: np.ndarray) -> float:
    """The standard error of the difference of two means: how much that difference
    moves by chance alone, given the spread and the number of runs of each."""
    return float(np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b)))


def versus(logs: dict[str, list[StudyLog]], reference: str = "random") -> pd.DataFrame:
    """Per sampler: its mean best score minus that of `reference`, in standard errors."""
    ref = np.array([values(log).max() for log in logs[reference]])
    rows = []
    for sampler, per_seed in logs.items():
        if sampler == reference:
            continue
        best = np.array([values(log).max() for log in per_seed])
        se = standard_error(best, ref)
        rows.append(
            {
                "sampler": sampler,
                f"minus {reference}": best.mean() - ref.mean(),
                "standard error": se,
                "in standard errors": (best.mean() - ref.mean()) / se
                if se > 0
                else np.nan,
            }
        )
    return pd.DataFrame(rows).set_index("sampler")


display(
    versus(round1)
    .style.format("{:+.4f}")
    .format("{:.4f}", subset=["standard error"])
    .format("{:+.1f}", subset=["in standard errors"])
    .set_caption("round 1: every sampler against random")
)

**On the head, the samplers lie within about half a point of each other.** The best
search on average is TPE (0.776), then random (0.774), grid and the population (both
0.770). Against random, TPE is +0.2 points, less than one standard error: no difference
you can tell from chance. Grid and the population are about 0.4 points below random, about
one and a half standard errors: a hint, not a result. Every sampler, every seed, ends
between 0.766 and 0.785.

The second table shows what each could draw: grid tried 3 learning rates in 240 trials,
random and TPE 240 different ones, the population 112 (a child copies its learning rate from
a parent, so values repeat).

What this says about the space, more than about the samplers: on the head of a frozen
`dinov2-small`, there is little to find. The four samplers drew very different trials and
ended in the same narrow band. A search that cannot separate its samplers is reporting that
the space is nearly flat, and that is information: more search on this space will not buy
much.

Where the samplers drew, for one seed. Every dot is a trial: its position is the trial's
number and the learning rate it drew, its colour the validation accuracy, its size the
depth.

In [ ]:
fig, axes = plt.subplots(1, len(SAMPLERS), figsize=(15, 3.6), sharey=True)
for ax, (sampler, per_seed) in zip(axes, round1.items()):
    trials = per_seed[0].trials(optuna.trial.TrialState.COMPLETE)
    points = ax.scatter(
        [t.number + 1 for t in trials],
        [t.params["lr"] for t in trials],
        c=[t.value for t in trials],
        cmap="viridis",
        vmin=0.6,
        vmax=0.8,
        s=[12 * t.params["depth"] ** 2 for t in trials],
    )
    ax.set_yscale("log")
    ax.set_title(f"{sampler}, seed {SEEDS[0]}")
    ax.set_xlabel("trial")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("lr drawn (log axis)")
fig.colorbar(points, ax=axes, label="validation accuracy")
fig.suptitle(
    "round 1: the lr every trial drew, in the order the trials ran (dot size = depth)"
)
plt.show()

**Grid** draws on three horizontal lines, one per learning rate in its lattice, and in no
order you can read: it does not look at the colours. **Random** scatters over the whole log
range from the first trial to the last, the same density everywhere, including the region
around `1e-2` where its trials score worst. **TPE** starts the same way (its first 10 trials
are random's first 10), and after about trial 15 its draws gather between about `2e-4` and `2e-3`, mostly with
the smallest depth, where the bright dots are. **The population** shares its first 8 trials
with random (same seed), and from the second generation on its trials sit on a few
learning rates, near those of the best parents, with an occasional mutation off the line.

That is the difference in what they know, in one picture: grid and random never change
where they draw; TPE and the population move to where the good trials were.

### Round 2: put the encoder in the space

The encoder is an axis that moves the score a lot (lesson 1 measured a gap of more than ten
points between DINOv2 and resnet-18 on DTD). So let the search choose it.

**Question:** does a search get better when an axis that matters is added to its space?

**Before you run it, write down:** for each sampler, will the best score in round 2 be
higher, equal or lower than in round 1, and why?

In [ ]:
round2 = race("encoder")
display(
    best_table(round2)
    .style.format("{:.4f}")
    .set_caption(
        f"round 2, encoder in the space: best validation accuracy of a search, over {len(SEEDS)} seeds"
    )
)
display(
    drawn(round2).style.set_caption(
        "round 2: what each sampler drew, all seeds together"
    )
)
display(
    versus(round2)
    .style.format("{:+.4f}")
    .format("{:.4f}", subset=["standard error"])
    .format("{:+.1f}", subset=["in standard errors"])
    .set_caption("round 2: every sampler against random")
)

**Every sampler got worse, and by different amounts.** The best score fell from round 1 to
round 2 for all four (the size of each fall, against chance, is measured in a moment). The
second table shows why: grid and random spread their trials evenly over the three encoders,
74 to 88 trials each of 240, so about two thirds of their budget went to `resnet-18` and
`mobilenet_v2`. TPE put 160 of its 240 trials on `dinov2-small`, the population 135. Against
random, TPE is now ahead by more than a point, at about two and a half standard errors.

Note grid in this table: Optuna's `GridSampler` walks its lattice in a shuffled order, so a
grid cut off at 40 of 135 points still tried all three encoders. A grid written as nested
`for` loops with the encoder outermost has 45 points per encoder, so cut off after 40 it
would have tried one encoder only.

What did the samplers find? First, every trial of round 2 grouped by encoder: is there a
difference between the encoders for a search to find at all? Then the winner of every single
search.

In [ ]:
trials2 = pd.DataFrame(
    [
        {
            "sampler": sampler,
            "encoder": t.user_attrs["config"]["encoder"],
            "val": t.value,
        }
        for sampler, per_seed in round2.items()
        for log in per_seed
        for t in log.trials(optuna.trial.TrialState.COMPLETE)
    ]
)
display(
    trials2.groupby("encoder")["val"]
    .agg(trials="count", worst="min", median="median", best="max")
    .sort_values("best", ascending=False)
    .style.format("{:.4f}", subset=["worst", "median", "best"])
    .set_caption(
        "round 2, every trial of every sampler: validation accuracy per encoder"
    )
)

table = winners(round2)
display(
    table.style.format({"val": "{:.4f}", "lr": "{:.1e}"})
    .hide(axis="index")
    .set_caption("round 2: the winner of every search")
)

**The encoder axis has a lot in it.** The best `dinov2-small` trial scores 0.785; the best
`mobilenet_v2` trial 0.636 and the best `resnet-18` trial 0.617, about fifteen points lower.
Even the median `dinov2-small` trial (0.741) is ten points above the best trial of either
other encoder.

**Every one of the 24 searches picked `dinov2-small`**, and this time that says something:
every sampler, grid included, drew all three encoders (the table of draws above), so each
search had the choice and made it. The winners also agree on the head: depth 1 in 19 of 24,
and a learning rate between `1e-4` and `2.5e-3`.

### Did the encoder axis make the search better or worse?

The best score of a search moves from seed to seed, so a difference between two rounds
means something only when it is large compared to that movement. The measure for that,
from lesson 1's `compare`, is the **standard error of the difference**: with $n$ searches
per round and spreads $s_1$ and $s_2$, it is $\sqrt{s_1^2/n + s_2^2/n}$, the amount the
difference between the two means moves by chance alone. A difference of two standard errors
or more is unlikely to be chance; one of less than one is what chance gives all the time.

In [ ]:
rows = []
for sampler in SAMPLERS:
    one = np.array([values(log).max() for log in round1[sampler]])
    two = np.array([values(log).max() for log in round2[sampler]])
    se = standard_error(two, one)
    rows.append(
        {
            "sampler": sampler,
            "round 1": one.mean(),
            "round 2": two.mean(),
            "difference": two.mean() - one.mean(),
            "standard error": se,
            "in standard errors": (two.mean() - one.mean()) / se if se > 0 else np.nan,
        }
    )
display(
    pd.DataFrame(rows)
    .set_index("sampler")
    .style.format("{:.4f}")
    .format("{:+.1f}", subset=["in standard errors"])
    .set_caption("best validation accuracy: round 2 against round 1, per sampler")
)

The difference in standard errors, per sampler:

- **random**: 1.1 points lower with the encoder in the space, at more than three standard
  errors. Random knows nothing about the trials so far, so the two thirds of its budget it
  spends on the weak encoders is simply gone: about 12 trials of 40 were left for the
  encoder that can win, where round 1 had all 40.
- **grid** and **the population**: about 0.4 points lower, at about two standard errors.
  Grid spends its budget the way random does; the population moves to `dinov2-small` but
  only between generations.
- **TPE**: 0.2 points lower, at about half a standard error, so no difference you can tell
  from chance. TPE noticed after its first trials that two encoders score badly and stopped
  drawing them.

So adding an axis that matters made no search better here, and how much it cost depends on
what the sampler knows. The encoder axis is worth fifteen points, but the best value of it
was already in round 1; putting the other two values in the space only added regions that
cannot win, and a sampler that does not learn pays for every trial it spends there.

> **A search space is a claim about where the answer might be.** A region you already know
> is bad does not make the search more thorough; with a fixed budget it takes trials away
> from the region that matters, unless the sampler learns to leave it.

### Best-so-far: read the shape

A search's **best-so-far curve** is, after every trial, the best score of the trials up to
then. It can only go up. Its *shape* says what more budget would buy, which the final
number does not:

- a curve that is **still climbing** at the last trial says the budget was the limit: more
  trials would likely find more;
- a curve that went **flat** early says the space is exhausted for this sampler: more trials
  of the same search buy nothing, and what would help is another space, not a longer search.

In [ ]:
STYLE = {
    "grid": {"color": "tab:blue"},
    "random": {"color": "tab:orange", "ls": "--"},
    "tpe": {"color": "tab:green"},
    "population": {"color": "tab:red", "ls": ":"},
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, (logs, title) in zip(
    axes, [(round1, "round 1: head only"), (round2, "round 2: head + encoder")]
):
    for sampler, per_seed in logs.items():
        curves = [np.maximum.accumulate(values(log)) for log in per_seed]
        n = min(len(c) for c in curves)
        curves = np.array([c[:n] for c in curves])
        mean, std = curves.mean(0), curves.std(0)
        x = np.arange(1, n + 1)
        ax.plot(x, mean, label=sampler, **STYLE[sampler])
        ax.fill_between(
            x, mean - std, mean + std, alpha=0.12, color=STYLE[sampler]["color"]
        )
    ax.set_title(title)
    ax.set_xlabel("trials run")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("best validation accuracy so far")
# the first few trials of a search lie below; the shape is up here
axes[0].set_ylim(0.70, 0.79)
axes[0].legend()
fig.suptitle(
    f"best-so-far per sampler: line = mean over {len(SEEDS)} seeds, band = their std"
)
plt.tight_layout()
plt.show()

**Read the start first.** For the first trials the lines of random, TPE and the population
lie on top of each other: with the same seed, TPE's 10 random startup trials and the
population's first generation of 8 are the trials random draws. Where a line leaves random's,
the sampler starts to use what it knows.

**Round 1:** every curve is within about a point of its end after 10 trials, and the last 20
trials add less than a point to any of them (most to random, 0.8 points, least to the
population, almost nothing). Grid starts slowest, since its early draws include the
`lr=1e-4` and `lr=1e-2` lines, and catches up by trial 21. This space is exhausted early: a
longer search on it buys little, whichever sampler runs it.

**Round 2:** the curves start lower, because many early trials land on a weak encoder, and
climb for longer. TPE leaves random after trial 10 and is the highest line from about trial
18 on. Random and the population are still rising in the last ten trials (the population
gains 0.7 points between trial 35 and 40); grid is flat from trial 28.

What would mean the opposite: a curve still climbing steeply at the last trial says "give
this search more trials"; a curve flat from trial 10 says "give it another space". Round 1
is the second kind; in round 2, random and the population are mildly the first kind, and
TPE, which learned to leave the weak encoders alone, is close to the second.

## 3.6 The number you may report

The winner of round 2 is the best of all its searches, each trial one run of 5 epochs, scored
on validation. Its validation score is the number the search *chose by*, and two things
separate it from a number you may report:

1. **Selection.** It is the maximum over hundreds of trials. The same configuration trained
   again with another seed scores, on the same split and for the same epochs, what that
   configuration scores, not what its luckiest run scored.
2. **Split and length.** The search looked at validation only, for 5 epochs. The reported
   number comes from test, which the search never saw, at a length you can defend.

Each is measured on its own below: the winner is retrained over the same seeds for up to
`FINAL_EPOCHS` epochs, with validation and test accuracy after every epoch.

**Before you run it, write down:** will the retrained winner, on validation after 5
epochs, score higher, equal or lower than its score in the search?

In [ ]:
everything = winners(round2)
top = everything.loc[everything["val"].idxmax()]
WINNER = {
    k: (v.item() if hasattr(v, "item") else v)
    for k, v in top[["encoder", "width", "depth", "lr"]].items()
}
print(
    f"winner of all round-2 searches: {top['sampler']} seed {top['seed']}, "
    f"val {top['val']:.4f} after {SEARCH_EPOCHS} epochs"
)
print(
    "config:",
    {k: round(v, 5) if isinstance(v, float) else v for k, v in WINNER.items()},
)

start = time.time()
retrain = [train_head(WINNER, seed, FINAL_EPOCHS, on=("val", "test")) for seed in SEEDS]
print(
    f"retrained over {len(SEEDS)} seeds for {FINAL_EPOCHS} epochs in {time.time() - start:.0f}s"
)

val = np.array([r["val"] for r in retrain])  # (seeds, epochs)
test = np.array([r["test"] for r in retrain])

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(1, FINAL_EPOCHS + 1)
for curve, label in [(val, "validation"), (test, "test")]:
    ax.plot(x, curve.mean(0), label=label)
    ax.fill_between(
        x, curve.mean(0) - curve.std(0), curve.mean(0) + curve.std(0), alpha=0.2
    )
ax.axvline(
    SEARCH_EPOCHS, color="k", ls=":", lw=1, label=f"the search's {SEARCH_EPOCHS} epochs"
)
ax.axhline(
    top["val"], color="crimson", ls="--", lw=1, label="the winner's score in the search"
)
ax.set_xlabel("epoch")
ax.set_ylabel("accuracy")
ax.set_title(f"the winner retrained over {len(SEEDS)} seeds: mean and std per epoch")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

**The retrained winner does not reach its own score in the search.** The dashed line is the
0.785 the winner scored in the search, on validation after 5 epochs, with torch seed 0. Over
six seeds, the same configuration on the same split after 5 epochs scores 0.772 on average.
Seed 0 is one of the six and gives 0.785 again; the other five are lower. The search kept the
configuration *and* its lucky seed.

**The length:** both curves climb steeply for 4 to 5 epochs and are flat from about epoch 8
to 30, on validation as on test. The flat part is where a reported number belongs, and the
length is chosen on the **validation** curve, not on test: `REPORT_EPOCHS = 20` lies well
inside it, and nothing between epoch 10 and 30 moves the validation mean by more than its
spread.

The steps from the search's number to the one you report, one change at a time:

In [ ]:
REPORT_EPOCHS = param(20, test=2)


def at(curves: np.ndarray, epochs: int) -> str:
    return (
        f"{curves[:, epochs - 1].mean():.4f} +- {curves[:, epochs - 1].std(ddof=1):.4f}"
    )


rows = [
    (
        f"search: the best of all trials, one seed, val, {SEARCH_EPOCHS} epochs",
        f"{top['val']:.4f}",
    ),
    (
        f"retrained: {len(SEEDS)} seeds, val, {SEARCH_EPOCHS} epochs",
        at(val, SEARCH_EPOCHS),
    ),
    (
        f"retrained: {len(SEEDS)} seeds, val, {REPORT_EPOCHS} epochs",
        at(val, REPORT_EPOCHS),
    ),
    (
        f"retrained: {len(SEEDS)} seeds, test, {SEARCH_EPOCHS} epochs",
        at(test, SEARCH_EPOCHS),
    ),
    (
        f"retrained: {len(SEEDS)} seeds, test, {REPORT_EPOCHS} epochs",
        at(test, REPORT_EPOCHS),
    ),
]
print(f"{'':52s} accuracy")
for label, number in rows:
    print(f"{label:52s} {number}")

runs = StudyLog.runs(
    name="lesson3-algorithm-runs", choices={"encoder": ENCODERS, "dataset": ["dtd"]}
)
runs.log(
    name="round-2 winner, retrained",
    role="final",
    results=Scalars(accuracy=list(test[:, REPORT_EPOCHS - 1])),
    dataset="dtd",
    params={"dataset": "dtd", **WINNER, "epochs": REPORT_EPOCHS},
    attrs={"split": "test", "found_by": str(top["sampler"])},
)

Read the table one row at a time; each row changes one thing:

1. **Selection** (row 1 to row 2): 0.785 to 0.772 ± 0.010. More than a point, all of it
   from choosing the best of 960 single runs. This is the optimism of a search winner, and
   it is the largest step in the table.
2. **Length** (row 2 to row 3, validation, 5 to 20 epochs): 0.772 to 0.774, inside the
   spread. On this head, 5 epochs was already close to done.
3. **Split** (row 3 to row 5, validation to test, 20 epochs): 0.774 to 0.778, inside the
   spread. Test is 752 other images, and on them the winner scores about the same.

**The number to report is 0.778 ± 0.006: the winner retrained for 20 epochs over six seeds,
scored on test**, which no search and no choice in this notebook looked at. It is lower
than the 0.785 the search showed, and that gap is what the search's optimism was here. The
run is written to the study `lesson3-algorithm-runs` with role `final`.

That the test score here lies slightly *above* validation is a property of these 752 rows,
not a rule: the order of the two can come out either way. What is a rule is that the
search's own number lies above what its winner gives again, because it was chosen as a
maximum.

## What you now know

- **What each sampler knows.** Grid knows a lattice and its cost is the product of the axes
  (45 points for the head, 135 with the encoder). Random draws every axis afresh and gives
  every trial a new value (240 learning rates against grid's 3), but never learns what it
  has seen. TPE models which values the good trials had and moves its draws there, after
  10 random trials. A population keeps the best 8 trials and makes children from them; it
  learns between generations and needs several of them.
- **A flat search is information.** On the head of `dinov2-small` the four samplers ended
  within half a point and within about one and a half standard errors of random: the space
  has little to find, and no sampler changes that.
- **The space matters more than the sampler.** Adding two encoders that are fifteen points
  worse made every search worse; it cost random most (more than three standard errors) and
  TPE least (half of one), because TPE learned to stop drawing them.
- **Read the shape of the best-so-far curve.** Flat early: change the space. Still climbing:
  more budget would help.
- **Report the test number of a retrained winner.** The search's 0.785 was the maximum over
  960 runs; the same configuration over six seeds gives 0.772 on validation and 0.778 on
  test at 20 epochs. The step that mattered was selection, not split or length.

| sampler | what it knows | removes this shortcoming of the one before | where it failed here |
|---|---|---|---|
| grid | a lattice per axis | | 3 learning rates only; spends evenly on bad encoders |
| random | the distribution per axis | grid's lattice: a new value of every axis in every trial | spends evenly on bad encoders: the largest loss in round 2 |
| TPE | which values the good trials had | random's blindness to the trials so far | nothing to learn on the flat head space |
| population (NSGA-II) | the best 8 trials so far | random forgets a good value after its trial | learns per generation: 5 generations in 40 trials |

Notebook 04 takes the budget question further: stop the trials that are clearly losing
after a few epochs (a pruner), and read the best trial against a second axis, the number of
parameters.

## Exercises

**E1: give the population room.** Run round 2 again with `BUDGET = 120` and
`population_size=20` for the population (six generations). Keep random, grid and TPE at the
same budget: equal trials is the control. Does the population catch up with TPE? Read the
answer in standard errors.

**E2: how much budget did the encoder axis cost random?** Count, per random search of
round 2, the trials that drew `dinov2-small`, and compare its best-so-far curve on those
trials only with round 1's random curve at the same number of `dinov2-small` trials. If the
two lie on each other, what does that say about where round 2's loss came from?

**E3: a space with something in it.** Both rounds search a head that barely moves the
score. Train on 10% of the training rows and search over regularisation (weight decay, or
the `GaussianNoise` sigma of lesson 1). Write down first whether you expect the samplers to
separate now, and why.

**E4: TPE's start.** Set `n_startup_trials=20` and then `5` in `TPESampler`. What changes in
round 2, in the number of trials on `dinov2-small` and in the best score? What is the risk of
a TPE that starts modelling after very few trials?